# TII UAV — IMU-Only Characterization + Standalone Mamba

## Goal

This notebook investigates the IMU **on its own**, then asks the important multimodal question:

> **Can IMU provide information that resolves the acoustic ambiguities, especially class 2↔3 and class 3↔4?**

It deliberately does **not** modify or tune the multimodal AeroSync-Mamba architecture.

### We will measure

1. Raw IMU integrity and timing (file order, plus a file-order vs sorted-order comparison).
2. Accelerometer/gyroscope temporal patterns.
3. Vibration spectra and class-specific frequency structure.
4. Class vs mission vs day effects.
5. Which IMU features distinguish adjacent severities (with bootstrap CIs and day-sign consistency).
6. A standalone real-Mamba IMU classifier.
7. Day-out performance on unseen recording days (mission-level cluster-bootstrap CIs).
8. Error structure: especially 2↔3 and 3↔4, and the safety-relevant errors.
9. Audio/IMU complementarity, using the audio notebook's **saved** per-window probabilities.

### Lined up with `TII_Audio_Only_Analysis_Mamba.ipynb`

- **Missions:** the same 95 (4 excluded). Per class [18, 19, 18, 20, 20]; per day [10, 36, 17, 15, 17], with 12-29
  merged into 12-31.
- **Protocol:** the same day-out folds and the same inner-validation rule. Seeds 42 / 123 / 2024.
- **Model and training:** the same as the audio model. v24/v28 settings, real `mamba_ssm`, 2 blocks, d_state 16,
  d_conv 2, d_model 256. AdamW 1e-4, 20 epochs, batch 32, best-validation-macro-F1 checkpoint.
- **IMU windows:** v24/v28's sensor arm, 2.56 s / 1.28 s stride with Ts = 128 samples.
- **Saved outputs:** the same schema as the audio notebook, so a third notebook can load either one the same way.

### Important

The official TII dataset contains accelerometer, gyroscope and audio data for five propeller-damage classes.
`SensorCombined` is a synchronized/downsampled IMU stream. The timestamp is used for timing; we do not assume an IMU
sample rate from documentation. Samples are kept in **file order** (as in v24), because IMU timestamps are not
monotonic in every mission.

## Expected dataset structure

`DATA_ROOT` is found automatically. The setup cell uses an attached Kaggle dataset containing `Dataset/0..4` if there
is one. Otherwise it runs `git clone --depth 1` of github.com/tiiuae/UAV-Realistic-Fault-Dataset, into the same place
the audio notebook uses.

```text
DATA_ROOT/
  0/
    rosbag2_.../
      ...-SensorCombined.jsonl
  1/
  2/
  3/
  4/
```

The parser is written to handle the common PX4 `SensorCombined` field names:

- `timestamp`
- `accelerometer_m_s2[0:3]`
- `gyro_rad[0:3]`

If your JSONL uses a slightly different nesting/schema, run the schema-inspection cell before changing anything else.

In [ ]:
# ==============================================================================
# SETUP — environment, output paths, dataset (attached Kaggle dataset, else git clone --depth 1)
# ==============================================================================
from pathlib import Path
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # before the first cuBLAS call (determinism)
import ast, copy, csv, gc, glob, json, math, random, re, subprocess, sys, time, warnings, hashlib
from collections import Counter, defaultdict
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

IS_KAGGLE = Path("/kaggle/working").exists()
IS_COLAB = (not IS_KAGGLE) and ("google.colab" in sys.modules or Path("/content").exists())
if IS_KAGGLE:
    WORK, BASE = Path("/kaggle/working"), Path("/kaggle/working/tii")       # BASE = the audio notebook's clone location
elif IS_COLAB:
    WORK, BASE = Path("/content"), Path("/content/tii")
else:
    WORK, BASE = Path.cwd() / "imu_only_outputs", Path.cwd() / "tii"
WORK.mkdir(parents=True, exist_ok=True)
BASE.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git"


def _is_dataset_root(p):
    return p.is_dir() and all((p / str(c)).is_dir() for c in range(5))


_attached = []
for _pat in ("/kaggle/input/*/Dataset", "/kaggle/input/*/*/Dataset", "/kaggle/input/*/*/*/Dataset"):
    _attached += [Path(p) for p in sorted(glob.glob(_pat)) if _is_dataset_root(Path(p))]
if _attached:
    DATA_ROOT, DATA_SOURCE = _attached[0], "attached Kaggle dataset"
else:
    _repo = BASE / "UAV-Realistic-Fault-Dataset"
    if not _is_dataset_root(_repo / "Dataset"):
        print(f"DATA_ROOT not found as an attached dataset — cloning {REPO_URL} (--depth 1) into {_repo} ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(_repo)], check=True)
    DATA_ROOT, DATA_SOURCE = _repo / "Dataset", "git clone --depth 1"
assert _is_dataset_root(DATA_ROOT), f"{DATA_ROOT} does not contain class folders 0..4"

OUT_DIR = WORK / "imu_only"                                   # parse cache
OUT_DIR.mkdir(parents=True, exist_ok=True)
IMU_PROBS_DIR = WORK / "imu_only_probs"                       # day_out_{fold}_{seed}.npz — the audio notebook's schema
IMU_RESULTS_CSV = WORK / "imu_only_results.csv"
IMU_MISSION_CSV = WORK / "imu_only_mission_predictions.csv"
IMU_PROBS_DIR.mkdir(parents=True, exist_ok=True)

# Project repo (public): used ONLY for (a) the saved audio day-out probabilities (section 16) and (b) the audio notebook
# (settings cross-check). Deliberately NOT searched for version 24/27/28, so the inner-validation rule stays the one
# the audio run used (its seed-2026 fallback). A failed clone is not fatal: section 16 then reports "no audio reference".
PROJECT_REPO_URL = "https://github.com/Sainava/AeroSync-Mamba.git"
PROJECT_REPO = BASE / "AeroSync-Mamba"
if not (PROJECT_REPO / ".git").exists():
    print(f"Cloning {PROJECT_REPO_URL} (--depth 1) into {PROJECT_REPO} ...")
    _r = subprocess.run(["git", "clone", "--depth", "1", PROJECT_REPO_URL, str(PROJECT_REPO)], capture_output=True, text=True)
    if _r.returncode != 0:
        print(f"  project repo clone FAILED (non-fatal): {(_r.stderr or _r.stdout)[-300:]}")
REPO_AUDIO_NB = PROJECT_REPO / "Audio Analysis " / "TII_Audio_Only_Analysis_Mamba.ipynb"
REPO_AUDIO_PROBS = PROJECT_REPO / "Audio Analysis " / "Analysis" / "PT and npz"

print("DATA_ROOT:", DATA_ROOT, f"({DATA_SOURCE})")
print("outputs:", WORK, "| cache:", OUT_DIR, "| per-window probs:", IMU_PROBS_DIR)
print("project repo:", PROJECT_REPO, "| audio probs in repo:", len(list(REPO_AUDIO_PROBS.glob("day_out_*.npz"))) if REPO_AUDIO_PROBS.is_dir() else 0, "files")

In [ ]:
# ==============================================================================
# REAL mamba-ssm (no fallback): install if missing, then import; a failure raises immediately
# ==============================================================================
import importlib
import importlib.util
import torch


def _pip(args):
    print("  pip install " + " ".join(args))
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    if r.returncode != 0:
        print((r.stderr or r.stdout)[-2000:])
    return r.returncode == 0


if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. Real mamba-ssm needs a GPU: enable a GPU accelerator and re-run. "
                       "There is deliberately no CPU fallback in this notebook.")
if importlib.util.find_spec("mamba_ssm") is None:
    print(f"Installing mamba-ssm (torch {torch.__version__}, CUDA {torch.version.cuda}) ...")
    _ok = (_pip(["packaging", "ninja"]) and _pip(["causal-conv1d>=1.2.0", "--no-build-isolation"])
           and _pip(["mamba-ssm", "--no-build-isolation"]))
    importlib.invalidate_caches()
    if not _ok:
        raise RuntimeError("mamba-ssm installation FAILED (see pip output above). Check internet + GPU, restart, re-run.")
try:
    import mamba_ssm
    from mamba_ssm import Mamba
except Exception as exc:
    raise RuntimeError(f"`from mamba_ssm import Mamba` failed ({exc!r}). Restart the kernel after installation and "
                       "re-run; no fallback model is used.") from exc
MAMBA_CLASS = Mamba
DEVICE = torch.device("cuda")
print(f"mamba_ssm {getattr(mamba_ssm, '__version__', '?')}: {Mamba.__module__}.{Mamba.__name__} | torch {torch.__version__} | "
      f"GPU {torch.cuda.get_device_name(0)}")

In [ ]:
# ==============================================================================
# CONSTANTS, SEEDING, and the cross-check against TII_Audio_Only_Analysis_Mamba.ipynb
# ==============================================================================
SEEDS = [42, 123, 2024]
N_CLASSES = 5
CHANNELS = ["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]

# --- mission set: identical to the audio notebook ---
EXPECTED_EXCLUDED = {"rosbag2_2021_12_29-09_10_31", "rosbag2_2021_12_31-10_12_31",
                     "rosbag2_2021_12_31-09_58_13", "rosbag2_2021_12_31-10_04_22"}
EXPECTED_N_MISSIONS = 95
EXPECTED_CLASS_COUNTS = [18, 19, 18, 20, 20]
DAYS = ["2021-12-22", "2021-12-31", "2022-01-04", "2022-01-05", "2022-01-10"]
DAY_MERGE = {"2021-12-29": "2021-12-31"}
EXPECTED_DAY_COUNTS = [10, 36, 17, 15, 17]
SMALL_FOLD_DAY = "2021-12-22"
STEADY = (0.10, 0.90)

# --- windows: v24/v28 sensor arm (read from version 24/minor-project-v24.ipynb and version 28/minor-project-v28.ipynb) ---
WIN_S = 2.56                     # WINDOW_SEC
STRIDE_S = 1.28                  # STEP_SEC
TS = 128                         # TS_SENSOR: samples per window, chosen evenly BY INDEX among the window's file-order samples
MIN_SAMPLES_PER_WINDOW = 10      # MIN_SENSOR_SAMPLES_PER_WINDOW
SENSOR_STD_EPS = 1e-6

# --- model + training: v24 == v28 (sensor arm S1), identical to the audio notebook ---
D_MODEL = 256
DROPOUT = 0.15
MAMBA_NUM_BLOCKS = 2
MAMBA_D_STATE = 16
MAMBA_D_CONV = 2
MAMBA_EXPAND = 2
EXPECTED_MAMBA_PARAMS = 874_496
EPOCHS = 20                      # fixed; no early stopping (v28)
BATCH_SIZE = 32
LEARNING_RATE = 1e-4             # AdamW
WEIGHT_DECAY = 1e-4
LR_MIN = 1e-6                    # CosineAnnealingLR(T_max=EPOCHS, eta_min=LR_MIN), stepped per epoch
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 2
# checkpoint rule (v28): keep the epoch with the highest validation macro-F1 (strictly greater)

# --- day-out inner validation: the audio notebook's rule (v27 _split_inner_val if readable, else fixed seed 2026) ---
INNER_VAL_FRACTION = 0.15
INNER_VAL_SEED = 2027
INNER_VAL_FALLBACK_SEED = 2026
V27_INNER_VAL_SHA1 = "58e3c037c513ace2"

# --- uncertainty ---
BOOTSTRAP_RESAMPLES = 2000
BOOTSTRAP_SEED = 2028
TOP_EFFECTS = 15                 # section 9: bootstrap CI for the top 15 features by |d|
TOP_DAY_SIGN = 10                # section 9: day-sign consistency for the top 10
SIGN_CONSISTENT_MIN_DAYS = 4
MIN_PER_CLASS_FOR_DAY_SIGN = 2

# --- validation gate ---
DETERMINISM_EPOCHS = 2
DETERMINISM_FOLD = 1             # day-out fold used for the determinism check (test day 2021-12-31)
DETERMINISM_STOP = True          # a determinism FAIL stops the notebook before the day-out runs


def seed_all(seed):
    """Seed python `random`, numpy, torch CPU and every CUDA device. Called at the START of every run."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_generator(seed):
    g = torch.Generator()
    g.manual_seed(seed)
    return g


def worker_init_fn(worker_id):
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s)
    random.seed(s)


torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)
seed_all(SEEDS[0])


def loud(msg):
    print("🚨" * 50)
    for line in msg.split("\n"):
        print("🚨  " + line)
    print("🚨" * 50)


SETTINGS = ["SEEDS", "EXPECTED_N_MISSIONS", "EXPECTED_CLASS_COUNTS", "DAYS", "DAY_MERGE", "EXPECTED_DAY_COUNTS", "STEADY",
            "WIN_S", "STRIDE_S", "TS", "MIN_SAMPLES_PER_WINDOW", "D_MODEL", "DROPOUT", "MAMBA_NUM_BLOCKS", "MAMBA_D_STATE",
            "MAMBA_D_CONV", "MAMBA_EXPAND", "EXPECTED_MAMBA_PARAMS", "EPOCHS", "BATCH_SIZE", "LEARNING_RATE", "WEIGHT_DECAY",
            "LR_MIN", "GRAD_CLIP_NORM", "NUM_WORKERS", "INNER_VAL_FRACTION", "INNER_VAL_SEED", "INNER_VAL_FALLBACK_SEED",
            "BOOTSTRAP_RESAMPLES", "BOOTSTRAP_SEED"]
print("SETTINGS USED:")
for k in SETTINGS:
    print(f"  {k:<26} = {globals()[k]}")
print("Determinism: cudnn.deterministic=True, benchmark=False, use_deterministic_algorithms(True, warn_only=True), "
      "seed_all() at the start of every run, seeded DataLoader generator + worker_init_fn")


def find_file(fname, subdirs):
    cands = sorted(glob.glob(f"/kaggle/input/**/{fname}", recursive=True))
    for base in (Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent):
        cands += [str(base / s / fname) for s in subdirs] + [str(base / fname)]
    return next((Path(c) for c in cands if Path(c).is_file()), None)


def notebook_constants(path):
    """Top-level literal assignments (incl. tuple assignments) from a notebook's code cells."""
    out = {}
    for cell in json.loads(Path(path).read_text(encoding="utf-8")).get("cells", []):
        if cell.get("cell_type") != "code":
            continue
        src = "".join(cell["source"]) if isinstance(cell["source"], list) else cell["source"]
        try:
            tree = ast.parse("\n".join(l for l in src.split("\n") if not l.lstrip().startswith(("!", "%"))))
        except SyntaxError:
            continue
        for node in tree.body:
            if not (isinstance(node, ast.Assign) and len(node.targets) == 1):
                continue
            tgt = node.targets[0]
            pairs = [(tgt, node.value)] if isinstance(tgt, ast.Name) else (
                list(zip(tgt.elts, node.value.elts)) if isinstance(tgt, ast.Tuple) and isinstance(node.value, ast.Tuple) else [])
            for t, v in pairs:
                if isinstance(t, ast.Name) and t.id not in out:
                    try:
                        out[t.id] = ast.literal_eval(v)
                    except ValueError:
                        pass
    return out


_audio_nb = find_file("TII_Audio_Only_Analysis_Mamba.ipynb", ["Audio Analysis ", "Audio Analysis"]) or (
    REPO_AUDIO_NB if REPO_AUDIO_NB.is_file() else None)
_SHARED = [k for k in SETTINGS if k not in ("WIN_S", "STRIDE_S", "TS", "MIN_SAMPLES_PER_WINDOW")] + ["EXPECTED_EXCLUDED", "V27_INNER_VAL_SHA1"]
if _audio_nb is None:
    loud("TII_Audio_Only_Analysis_Mamba.ipynb is not attached, so it cannot be re-checked at run time.\n"
         "Using the settings transcribed from the revised audio notebook (2026-09-29), printed above.")
else:
    _ac = notebook_constants(_audio_nb)
    _diff = {k: (globals()[k], _ac.get(k)) for k in _SHARED if _ac.get(k) != globals()[k]}
    if _diff:
        loud(f"MISMATCH with the audio notebook ({_audio_nb}):\n{_diff}")
        raise RuntimeError("IMU settings do not line up with TII_Audio_Only_Analysis_Mamba.ipynb — stopping.")
    print(f"Cross-check vs {_audio_nb}: all {len(_SHARED)} shared settings identical ✅")

In [ ]:
# Discover missions and SensorCombined files; apply the project's 4-mission exclusion
def mission_day(name):
    d = name[8:18].replace("_", "-")          # rosbag2_YYYY_MM_DD-... -> YYYY-MM-DD
    return DAY_MERGE.get(d, d)


mission_rows = []
for class_dir in sorted(DATA_ROOT.iterdir()):
    if not class_dir.is_dir() or not class_dir.name.isdigit():
        continue
    cls = int(class_dir.name)
    for mission_dir in sorted(class_dir.iterdir()):
        if not mission_dir.is_dir() or mission_dir.name.startswith("."):
            continue
        candidates = sorted(mission_dir.glob("*-SensorCombined.jsonl"))
        mission_rows.append({"mission": mission_dir.name, "class": cls, "day": mission_day(mission_dir.name),
                             "path": str(candidates[0]) if candidates else None})

all_missions = pd.DataFrame(mission_rows)
missions = all_missions[~all_missions["mission"].isin(EXPECTED_EXCLUDED)].reset_index(drop=True)
print(f"mission folders: {len(all_missions)} | excluded (project-wide list): {sorted(EXPECTED_EXCLUDED)}")
print("missions kept:", len(missions))
print(missions.groupby("class").size().rename("missions").to_frame().T.to_string())
print("\nday x class:")
print(pd.crosstab(missions["day"], missions["class"]).reindex(DAYS).to_string())
missions.head()

**Why 4 missions are excluded.** The IMU data of these 4 missions is **not** known to be corrupted:
- rosbag2_2021_12_29-09_10_31
- rosbag2_2021_12_31-10_12_31
- rosbag2_2021_12_31-09_58_13
- rosbag2_2021_12_31-10_04_22

Only their *audio* `recording_started_time` metadata is corrupt: its offset / mission duration is ≥ 0.5. They are
excluded here **solely so that the mission set matches every other notebook in the project**, including the audio-only
notebook. That leaves 95 missions and makes the day-out folds and the mission-level comparison in section 16
identical. The single remaining 2021-12-29 mission is merged into the 2021-12-31 day group.

## 1. Inspect one raw SensorCombined record

Do this before parsing the entire dataset. This prevents silently assuming the wrong JSON schema.


In [ ]:
sample_path = Path(missions.iloc[0]["path"])
with sample_path.open("r") as f:
    first_line = f.readline()

sample_obj = json.loads(first_line)
print(json.dumps(sample_obj, indent=2)[:5000])


In [ ]:
def unwrap_message(obj):
    # Handle common ROS/PX4 JSONL wrappers.
    if isinstance(obj, dict):
        for key in ("msg", "message", "data"):
            if key in obj and isinstance(obj[key], dict):
                return obj[key]
    return obj

msg = unwrap_message(sample_obj)
print("Top-level keys:", list(msg.keys())[:100])


## 2. Full IMU parser

We retain exactly six physical channels:

`acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z`

No audio values are read here.

The timestamp is retained so we can measure the actual IMU sampling interval, gaps and mission duration.

**Sample order.** The parser returns samples in **file order** and never sorts them. v24 and the rest of the modelling
pipeline read samples in file order, because IMU timestamps are not monotonic in every mission. Each mission also gets
a timestamp-**sorted** copy. That copy is used only in the file-order vs sorted-order comparison (cell 3b).
Everything else (timing, plots, spectra, windows, training) uses file order.

In [ ]:
def get_vec(d, candidates):
    for key in candidates:
        if key in d:
            v = d[key]
            if isinstance(v, (list, tuple)) and len(v) >= 3:
                return np.asarray(v[:3], dtype=np.float64)
        # Some JSON encodings may flatten [0], [1], [2].
        vals = []
        ok = True
        for i in range(3):
            k = key.format(i=i)
            if k in d:
                vals.append(d[k])
            else:
                ok = False
                break
        if ok:
            return np.asarray(vals, dtype=np.float64)
    return None

def parse_imu_file(path):
    """Rows [timestamp_us, acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z] in FILE ORDER (never sorted here)."""
    rows = []
    bad = 0
    with open(path, "r") as f:
        for line in f:
            try:
                obj = unwrap_message(json.loads(line))
                ts = obj.get("timestamp", obj.get("time_stamp", None))
                acc = get_vec(obj, [
                    "accelerometer_m_s2",
                    "accelerometer_m_s2[{i}]",
                    "accelerometer",
                    "acceleration",
                ])
                gyro = get_vec(obj, [
                    "gyro_rad",
                    "gyro_rad[{i}]",
                    "gyro",
                    "angular_velocity",
                ])
                if ts is None or acc is None or gyro is None:
                    bad += 1
                    continue
                rows.append([float(ts), *acc.tolist(), *gyro.tolist()])
            except Exception:
                bad += 1

    arr = np.asarray(rows, dtype=np.float64).reshape(-1, 7)
    return arr, bad

arr, bad = parse_imu_file(sample_path)
print("shape:", arr.shape, "bad/unusable lines:", bad)
print("first row:", arr[0] if len(arr) else None)

In [ ]:
# Parse/cache all kept missions. The cache stores FILE ORDER; the sorted copy is derived on load.
cache_dir = OUT_DIR / "cache_fileorder"        # new name: an older cache in OUT_DIR/cache held SORTED arrays
cache_dir.mkdir(exist_ok=True)

imu_data = {}
integrity = []

for i, r in missions.iterrows():
    cache_path = cache_dir / f"{r['mission']}.npz"
    if r["path"] is None:
        t, x, bad = np.empty(0), np.empty((0, 6)), 0
    elif cache_path.exists():
        z = np.load(cache_path)
        x = z["x"]
        t = z["t"]
        bad = int(z["bad"])
    else:
        raw, bad = parse_imu_file(Path(r["path"]))
        t, x = raw[:, 0], raw[:, 1:]
        np.savez_compressed(cache_path, t=t, x=x, bad=bad)

    order = np.argsort(t, kind="stable")
    imu_data[r["mission"]] = {"t": t, "x": x, "t_sorted": t[order], "x_sorted": x[order],
                              "class": int(r["class"]), "day": r["day"]}

    dt = np.diff(t) / 1e6                       # FILE ORDER: an inversion shows up as a negative interval
    integrity.append({
        "mission": r["mission"],
        "class": int(r["class"]),
        "day": r["day"],
        "n": len(t),
        "duration_s": (t[-1]-t[0])/1e6 if len(t)>1 else np.nan,
        "median_dt_ms": np.median(dt)*1e3 if len(dt) else np.nan,
        "p01_dt_ms": np.percentile(dt,1)*1e3 if len(dt) else np.nan,
        "p99_dt_ms": np.percentile(dt,99)*1e3 if len(dt) else np.nan,
        "max_dt_ms": np.max(dt)*1e3 if len(dt) else np.nan,
        "n_negative_dt": int((dt < 0).sum()),
        "bad_lines": bad,
        "finite": bool(len(t) > 0 and np.isfinite(t).all() and np.isfinite(x).all()),
    })

integrity = pd.DataFrame(integrity)
print(integrity.groupby("class")[["n","duration_s","median_dt_ms","max_dt_ms"]].median())
print("total bad lines:", integrity.bad_lines.sum())

## Validation gate (part A — data)
Runs after parsing and before any plotting or modelling. Each check prints PASS or FAIL. **If any check FAILs, the
notebook stops here with a loud warning, before section 3.**

The determinism check needs the model and the folds, so it runs as **part B** of the gate, immediately before the
day-out runs. It also stops on FAIL.

In [ ]:
# ==============================================================================
# VALIDATION GATE — part A (data)
# ==============================================================================
GATE = []


def gate(name, ok, detail=""):
    ok = bool(ok)
    GATE.append((name, ok, detail))
    print(f"[GATE] {'PASS' if ok else 'FAIL'} — {name}" + (f" | {detail}" if detail else ""))
    return ok


_cls = np.bincount(missions["class"], minlength=N_CLASSES).tolist()
_days = [int((missions["day"] == d).sum()) for d in DAYS]
gate("mission and class counts", len(missions) == EXPECTED_N_MISSIONS and _cls == EXPECTED_CLASS_COUNTS,
     f"{len(missions)} missions, per class {_cls} (expected {EXPECTED_N_MISSIONS}, {EXPECTED_CLASS_COUNTS})")
gate("day counts", _days == EXPECTED_DAY_COUNTS and set(missions["day"]) == set(DAYS),
     f"{dict(zip(DAYS, _days))} (expected {EXPECTED_DAY_COUNTS}; 12-29 merged into 12-31)")
_bad_finite = [m for m in missions["mission"] if not (len(imu_data[m]["t"]) > 0 and np.isfinite(imu_data[m]["t"]).all()
                                                       and np.isfinite(imu_data[m]["x"]).all())]
gate("every mission's parsed array is non-empty and finite (checked per mission)", not _bad_finite,
     f"{len(missions) - len(_bad_finite)}/{len(missions)} OK" + (f"; FAILING: {_bad_finite}" if _bad_finite else ""))
_folders = set(all_missions["mission"])
_kept = set(missions["mission"])
_excl = _folders - _kept
gate("no mission excluded that shouldn't be, and no unexpected mission missing",
     _excl == EXPECTED_EXCLUDED and EXPECTED_EXCLUDED <= _folders and all_missions.loc[
         all_missions.mission.isin(_kept), "path"].notna().all(),
     f"excluded = {sorted(_excl)}; all 4 expected folders present: {EXPECTED_EXCLUDED <= _folders}; "
     f"every kept mission has a SensorCombined file: {all_missions.loc[all_missions.mission.isin(_kept), 'path'].notna().all()}")
gate("window ids/count per fold match across protocols", True, "N/A — day-out only (no matched-random protocol here)")
print("[GATE] determinism check -> part B, immediately before the day-out runs (needs the model and folds)")
if not all(ok for _, ok, _ in GATE):
    loud("VALIDATION GATE (part A) FAILED:\n" + "\n".join(f"{n}: {d}" for n, ok, d in GATE if not ok)
         + "\nStopping before section 3. Fix the data/paths and re-run.")
    raise RuntimeError("Validation gate (part A) failed — see the FAIL lines above.")
print("Validation gate part A: all PASS")

## 3. What is the actual IMU timing?

Unlike audio, `SensorCombined` has timestamps. Therefore we can derive the sampling interval directly from the data.

We should **not** carry over the audio uncertainty about 16 kHz here.

The statistics below use **file order**. If a mission's timestamps go backwards, its intervals include negative
values. Cell 3b compares file order with sorted order.

In [ ]:
print(integrity[["median_dt_ms","p01_dt_ms","p99_dt_ms","max_dt_ms"]].describe())

plt.figure(figsize=(8,4))
plt.hist(integrity["median_dt_ms"].dropna(), bins=30)
plt.xlabel("mission median IMU interval (ms)")
plt.ylabel("missions")
plt.title("IMU sampling interval across missions")
plt.grid(alpha=.25)
plt.show()


## 3b. Timestamp order — file order vs sorted order
For each mission this cell reports:
- the count and fraction of adjacent-sample timestamp inversions in file order (`t[i+1] < t[i]`);
- the largest backward jump;
- how the median / p01 / p99 sampling interval changes between file order and sorted order.

File order stays primary: windows and training use it, as v24 does. Nothing here changes the data used below.

In [ ]:
# ==============================================================================
# 3b. TIMESTAMP INVERSIONS AND FILE-ORDER vs SORTED-ORDER INTERVAL STATISTICS (per mission)
# ==============================================================================
_rows = []
for m in missions["mission"]:
    d = imu_data[m]
    dtf = np.diff(d["t"]) / 1e3                 # ms, file order
    dts = np.diff(d["t_sorted"]) / 1e3          # ms, sorted
    inv = dtf < 0
    q = lambda a: np.percentile(a, [50, 1, 99]) if len(a) else np.full(3, np.nan)
    (mf, p1f, p99f), (ms_, p1s, p99s) = q(dtf), q(dts)
    _rows.append({"mission": m, "class": d["class"], "day": d["day"], "n": len(d["t"]),
                  "inversions": int(inv.sum()), "inversion_frac": float(inv.mean()) if len(inv) else 0.0,
                  "max_backward_jump_ms": float(-dtf[inv].min()) if inv.any() else 0.0,
                  "median_dt_file_ms": mf, "median_dt_sorted_ms": ms_, "p01_dt_file_ms": p1f, "p01_dt_sorted_ms": p1s,
                  "p99_dt_file_ms": p99f, "p99_dt_sorted_ms": p99s,
                  "median_diff_ms": mf - ms_, "p01_diff_ms": p1f - p1s, "p99_diff_ms": p99f - p99s})
TS_ORDER = pd.DataFrame(_rows).set_index("mission")
NONMONO = TS_ORDER[TS_ORDER.inversions > 0].sort_values("inversion_frac", ascending=False)

print(f"Missions with non-monotonic timestamps in file order: {len(NONMONO)} / {len(TS_ORDER)}")
print(f"Total inversions: {int(TS_ORDER.inversions.sum()):,} of {int((TS_ORDER.n - 1).sum()):,} adjacent pairs "
      f"({100 * TS_ORDER.inversions.sum() / (TS_ORDER.n - 1).sum():.4f}%)")
if len(NONMONO):
    print("\nNon-monotonic missions (sorted by inversion fraction):")
    print(NONMONO[["class", "day", "n", "inversions", "inversion_frac", "max_backward_jump_ms", "median_dt_file_ms",
                   "median_dt_sorted_ms", "p01_dt_file_ms", "p01_dt_sorted_ms", "p99_dt_file_ms",
                   "p99_dt_sorted_ms"]].round(4).to_string())
    _rel = (NONMONO.median_diff_ms.abs() / NONMONO.median_dt_sorted_ms).max()
    print(f"\nPlainly: {len(NONMONO)} mission(s) have timestamps that go backwards in file order. Up to "
          f"{100 * NONMONO.inversion_frac.max():.3f}% of a mission's adjacent pairs are inverted, and the largest "
          f"backward jump is {NONMONO.max_backward_jump_ms.max():.2f} ms. The median sampling interval changes by at "
          f"most {100 * _rel:.2f}% between file and sorted order. p01 changes by up to "
          f"{NONMONO.p01_diff_ms.abs().max():.3f} ms and p99 by up to {NONMONO.p99_diff_ms.abs().max():.3f} ms.")
else:
    print("Plainly: every mission's timestamps are monotonic non-decreasing in file order. File order == sorted order "
          "for timing statistics.")
print("\nAll 95 missions (file − sorted differences, ms):")
print(TS_ORDER[["inversions", "median_diff_ms", "p01_diff_ms", "p99_diff_ms"]].describe().round(4).to_string())

## 4. Representative raw traces

We look at acceleration and angular velocity separately. The key question is whether fault severity changes the vibration pattern, amplitude, or frequency content.

Sections 4–9 use the **file-order** samples, the same arrays the model sees.

In [ ]:
def plot_trace(mission, start_frac=.40, duration_s=2.0):
    d = imu_data[mission]
    t = (d["t"] - d["t"][0]) / 1e6
    x = d["x"]
    start = int(len(t)*start_frac)
    t0 = t[start]
    keep = (t >= t0) & (t <= t0 + duration_s)

    fig, ax = plt.subplots(2,1,figsize=(12,6),sharex=True)
    for j, lab in enumerate(["acc_x","acc_y","acc_z"]):
        ax[0].plot(t[keep]-t0, x[keep,j], label=lab)
    for j, lab in enumerate(["gyro_x","gyro_y","gyro_z"], start=3):
        ax[1].plot(t[keep]-t0, x[keep,j], label=lab)
    ax[0].set_ylabel("m/s²")
    ax[1].set_ylabel("rad/s")
    ax[1].set_xlabel("time (s)")
    ax[0].legend(ncol=3); ax[1].legend(ncol=3)
    fig.suptitle(f"{mission} | class {d['class']}")
    plt.tight_layout()
    plt.show()

for mission in missions.groupby("class").first()["mission"]:
    plot_trace(mission)


## 5. Steady-flight IMU statistics

Use the same 10–90% mission convention as the audio analysis.

For each mission and channel we calculate:

- RMS / standard deviation
- mean
- peak-to-peak
- high-percentile absolute magnitude

This tells us whether the IMU signal is mainly an amplitude/severity signal.


In [ ]:
def steady_stats(mission):
    d = imu_data[mission]
    x = d["x"]
    n = len(x)
    a,b = int(.10*n), int(.90*n)
    xs = x[a:b]
    out = {"mission":mission, "class":d["class"]}
    for j, name in enumerate(["acc_x","acc_y","acc_z","gyro_x","gyro_y","gyro_z"]):
        v = xs[:,j]
        out[f"{name}_mean"] = np.mean(v)
        out[f"{name}_std"] = np.std(v)
        out[f"{name}_rms"] = np.sqrt(np.mean(v*v))
        out[f"{name}_p99abs"] = np.percentile(np.abs(v),99)
        out[f"{name}_ptp"] = np.ptp(v)
    return out

imu_stats = pd.DataFrame([steady_stats(m) for m in missions["mission"]])
display(imu_stats.groupby("class").median(numeric_only=True).round(4))


## 6. IMU vibration spectrum

For each mission we estimate a Welch-like spectrum from the steady-flight signal.

The frequency axis is valid here because the sampling interval comes from the recorded timestamps.

We will inspect both:

- accelerometer vibration
- gyroscope vibration

The important question is whether increasing propeller damage creates stronger/different vibration bands.


In [ ]:
from scipy.signal import welch

def mission_psd(mission, nperseg=2048):
    d = imu_data[mission]
    t = d["t"]
    x = d["x"]
    a,b = int(.10*len(x)), int(.90*len(x))
    xs = x[a:b]

    dt = np.median(np.diff(t[a:b])) / 1e6
    fs = 1.0 / dt
    spectra = []
    freqs = None
    for j in range(6):
        f, p = welch(xs[:,j], fs=fs, nperseg=min(nperseg,len(xs)), detrend="constant")
        freqs = f
        spectra.append(p)
    return freqs, np.asarray(spectra)

# Plot class means
fig, axes = plt.subplots(2,1,figsize=(12,9),sharex=True)
for cls in sorted(missions["class"].unique()):
    arrs = []
    for m in missions.loc[missions["class"]==cls,"mission"]:
        f,p = mission_psd(m)
        arrs.append(p)
    meanp = np.mean(arrs,axis=0)
    for j in range(3):
        axes[0].plot(f, 10*np.log10(meanp[j]+1e-12), alpha=.75, label=f"class {cls}" if j==0 else None)
        axes[1].plot(f, 10*np.log10(meanp[j+3]+1e-12), alpha=.75, label=f"class {cls}" if j==0 else None)
axes[0].set_xlim(0, min(5000, f.max()))
axes[1].set_xlim(0, min(5000, f.max()))
axes[0].set_title("Accelerometer class-mean PSD")
axes[1].set_title("Gyroscope class-mean PSD")
axes[1].set_xlabel("Hz")
axes[0].set_ylabel("log power"); axes[1].set_ylabel("log power")
axes[0].legend()
plt.tight_layout()
plt.show()


## 7. Frequency-band features

To test the physical pattern without committing to a neural architecture, summarize IMU energy into logarithmic frequency bands.

These features are descriptive only.

We will then ask:

- Which channels carry class information?
- Which bands change monotonically with severity?
- Is the signal mainly accelerometer or gyroscope?


In [ ]:
_trapz = getattr(np, "trapezoid", None) or np.trapz     # np.trapz is deprecated in NumPy 2

def band_features(mission, bands=None):
    if bands is None:
        bands = [(5,20),(20,50),(50,100),(100,200),(200,500),(500,1000),(1000,2000),(2000,5000)]
    f,p = mission_psd(mission)
    out={"mission":mission,"class":imu_data[mission]["class"]}
    for j,name in enumerate(["acc_x","acc_y","acc_z","gyro_x","gyro_y","gyro_z"]):
        for lo,hi in bands:
            mask=(f>=lo)&(f<hi)
            out[f"{name}_{lo}_{hi}"] = _trapz(p[j,mask], f[mask]) if mask.any() else np.nan
    return out

imu_band = pd.DataFrame([band_features(m) for m in missions["mission"]])
print(imu_band.shape)
display(imu_band.groupby("class").median(numeric_only=True).round(4))

## 8. Class effect vs day effect

Same idea as the audio analysis.

We want to know whether IMU contains:

- a strong fault-class signal,
- a strong recording-day signal,
- or mostly mission-specific variation.

The PCA is descriptive; the variance decomposition is interpreted cautiously because there are only five days.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

feature_cols = [c for c in imu_band.columns if c not in ("mission","class")]
X = imu_band[feature_cols].replace([np.inf,-np.inf],np.nan).fillna(0).values
Xz = StandardScaler().fit_transform(X)

pca = PCA(n_components=2).fit_transform(Xz)
day_map = missions.set_index("mission")["day"]          # YYYY-MM-DD, 12-29 merged into 12-31 (project-wide)
imu_band["day"] = imu_band["mission"].map(day_map)

fig, ax = plt.subplots(1,2,figsize=(13,5))
for cls in sorted(imu_band["class"].unique()):
    mask=imu_band["class"].values==cls
    ax[0].scatter(pca[mask,0],pca[mask,1],label=f"class {cls}",alpha=.8)
for day in sorted(imu_band["day"].dropna().unique()):
    mask=imu_band["day"].values==day
    ax[1].scatter(pca[mask,0],pca[mask,1],label=day,alpha=.8)
ax[0].set_title("IMU PCA — coloured by class")
ax[1].set_title("IMU PCA — coloured by day")
for a in ax:
    a.set_xlabel("PC1"); a.set_ylabel("PC2"); a.grid(alpha=.2)
ax[0].legend()
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

print("Explained variance:", PCA(n_components=5).fit(Xz).explained_variance_ratio_[:5])

## 9. Adjacent-severity IMU effect sizes

This is one of the most important sections.

We explicitly test:

- 0 vs 1
- 1 vs 2
- 2 vs 3
- 3 vs 4

If IMU separates 2↔3 and 3↔4 using **different features from audio**, it gives us evidence that IMU could rescue acoustic ambiguity.

**Uncertainty** (the same method as section 3b of the audio notebook):
- For the top 15 features per pair by |d|, a 95 % CI from a mission-level bootstrap: 2000 resamples of missions
  within each class.
- For the top 10, the number of the 5 days on which the class-mean difference has the same sign as d. Only days with
  ≥ 2 missions in both classes count. A feature is marked consistent if the sign holds on ≥ 4 of the 5 days.

In [ ]:
from scipy.stats import ttest_ind

def cohens_d(a,b):
    a=np.asarray(a); b=np.asarray(b)
    na,nb=len(a),len(b)
    va,vb=np.var(a,ddof=1),np.var(b,ddof=1)
    sp=np.sqrt(((na-1)*va+(nb-1)*vb)/(na+nb-2))
    return (np.mean(b)-np.mean(a))/sp if sp>0 else np.nan

rows=[]
for c in range(4):
    A=imu_band[imu_band["class"]==c]
    B=imu_band[imu_band["class"]==c+1]
    for feat in feature_cols:
        d=cohens_d(A[feat],B[feat])
        rows.append({"pair":f"{c} vs {c+1}","feature":feat,"d":d,"abs_d":abs(d)})
effects=pd.DataFrame(rows)

# ---- 95% mission-level bootstrap CI (top 15) and day-sign consistency (top 10) ----
_boot_rng = np.random.default_rng(BOOTSTRAP_SEED)
_y_m = imu_band["class"].values
_day_m = imu_band["day"].values


def boot_d(Xa, Xb):
    """Cohen's d (b - a) per column over BOOTSTRAP_RESAMPLES resamples of missions within each class."""
    ia = _boot_rng.integers(0, len(Xa), (BOOTSTRAP_RESAMPLES, len(Xa)))
    ib = _boot_rng.integers(0, len(Xb), (BOOTSTRAP_RESAMPLES, len(Xb)))
    A, B = Xa[ia], Xb[ib]
    na, nb = len(Xa), len(Xb)
    sp = np.sqrt(((na - 1) * A.var(1, ddof=1) + (nb - 1) * B.var(1, ddof=1)) / (na + nb - 2))
    with np.errstate(invalid="ignore", divide="ignore"):
        return (B.mean(1) - A.mean(1)) / sp


EFFECT_TABLES = {}
for c in range(4):
    pair = f"{c} vs {c+1}"
    top = effects[effects["pair"] == pair].dropna(subset=["d"]).sort_values("abs_d", ascending=False).head(TOP_EFFECTS)
    feats = top["feature"].tolist()
    Xa = imu_band.loc[_y_m == c, feats].values.astype(np.float64)
    Xb = imu_band.loc[_y_m == c + 1, feats].values.astype(np.float64)
    lo, hi = np.nanpercentile(boot_d(Xa, Xb), [2.5, 97.5], axis=0)
    out = top[["feature", "d"]].copy().reset_index(drop=True)
    out["CI low"], out["CI high"] = lo, hi
    same_days, consistent = [], []
    for k, feat in enumerate(feats):
        if k >= TOP_DAY_SIGN:
            same_days.append("")
            consistent.append("")
            continue
        same, elig = 0, 0
        for day in DAYS:
            sa = (_y_m == c) & (_day_m == day)
            sb = (_y_m == c + 1) & (_day_m == day)
            if sa.sum() >= MIN_PER_CLASS_FOR_DAY_SIGN and sb.sum() >= MIN_PER_CLASS_FOR_DAY_SIGN:
                elig += 1
                diff = imu_band.loc[sb, feat].mean() - imu_band.loc[sa, feat].mean()
                same += int(np.sign(diff) == np.sign(out.loc[k, "d"]))
        same_days.append(f"{same}/{elig}")
        consistent.append("✔ consistent" if same >= SIGN_CONSISTENT_MIN_DAYS else "")
    out["days same sign (top 10)"] = same_days
    out[f"consistent on >= {SIGN_CONSISTENT_MIN_DAYS} days"] = consistent
    EFFECT_TABLES[pair] = out
    print(f"\n{pair}  (d = class {c+1} − class {c}; CI = 95% mission-level bootstrap, {BOOTSTRAP_RESAMPLES} resamples within class)")
    display(out.round(3))

## 10. Window-level IMU representation

For the classifier we use timestamp-defined windows rather than an assumed sample rate. The settings are v24/v28's
sensor arm:

- 2.56 s window
- 1.28 s stride
- Ts = 128 samples per window
- 6 IMU channels
- at least 10 samples in a window

**File order.** Time is relative to the mission's first sample in file order, as in v24. A window holds the file-order
samples whose timestamp falls inside it, and 128 of them are chosen **evenly by index**. This is v24's sensor slicing,
which picks real samples; nothing is interpolated. Interpolating on the time axis (`np.interp`) is undefined when
timestamps go backwards, so it is not used.

Each window records an id `mission#NNNN`, its mission, class, day, start time and phase. The phase is start (centre
< 10 %), steady (10–90 %) or end (> 90 % of the mission duration). These are the audio notebook's fields.

The model sees temporal IMU structure directly.

In [ ]:
windows=[]
for _,r in missions.iterrows():
    m=r["mission"]; d=imu_data[m]
    t=(d["t"]-d["t"][0])/1e6          # FILE ORDER, seconds relative to the first sample (v24)
    x=d["x"]
    if len(t)<2 or t[-1]<WIN_S:
        continue
    dur=t[-1]

    starts=np.arange(0, dur-WIN_S+1e-9, STRIDE_S)
    i=0
    for s in starts:
        e=s+WIN_S
        idx=np.where((t>=s)&(t<=e))[0]                     # file-order positions inside the window
        if len(idx)<MIN_SAMPLES_PER_WINDOW:
            continue
        pick=idx[np.linspace(0,len(idx)-1,TS).round().astype(int)]   # v24: TS real samples, evenly by index
        rel=(s+WIN_S/2)/dur
        windows.append({"window_id":f"{m}#{i:04d}","mission":m,"class":int(r["class"]),"day":r["day"],
                        "start_s":float(s),"phase":"start" if rel<STEADY[0] else ("end" if rel>STEADY[1] else "steady"),
                        "n_samples":len(idx),"x":x[pick].astype(np.float32)})
        i+=1

print("windows:",len(windows))
_wpm = Counter(w["mission"] for w in windows)
assert len(_wpm) == len(missions), f"{len(missions) - len(_wpm)} missions produced no window"
assert all(w["x"].shape == (TS, 6) and np.isfinite(w["x"]).all() for w in windows)
assert len({w["window_id"] for w in windows}) == len(windows)
print(f"every window: shape ({TS}, 6), finite, one mission; windows per mission median {np.median(list(_wpm.values())):.0f} "
      f"(min {min(_wpm.values())}, max {max(_wpm.values())}); samples per window median "
      f"{np.median([w['n_samples'] for w in windows]):.0f}")

## 11. Train-only normalization

All normalization is fitted on training missions only within each fold.

This is important because otherwise mission/day information can leak into the test set.


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, classification_report, confusion_matrix, roc_auc_score
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

W=np.stack([w["x"] for w in windows]).astype(np.float32)
y=np.array([w["class"] for w in windows],dtype=np.int64)
groups=np.array([w["mission"] for w in windows])
days=np.array([w["day"] for w in windows])
phases=np.array([w["phase"] for w in windows])
window_ids=np.array([w["window_id"] for w in windows])
starts_s=np.array([w["start_s"] for w in windows])

print(W.shape, y.shape)
print(pd.crosstab(pd.Series(days, name="day"), pd.Series(y, name="class")).reindex(DAYS).to_string())

## 12. Real Mamba classifier

This is v28's `UnimodalMamba` sensor path (arm S1). The audio notebook uses the same model with its own encoder.

```text
6-channel IMU (Ts = 128)
    ↓
1D-CNN encoder: Conv1d(6→64, k5) → BN → GELU → Conv1d(64→256, k3) → BN → GELU
    ↓
+ modality embedding → LayerNorm
    ↓
2 real Mamba blocks (pre-norm residual; d_state 16, d_conv 2, expand 2)
    ↓
attention pooling
    ↓
classifier: Dropout(0.15) → Linear(256→128) → GELU → Linear(128→5)
```

No audio, no LACA, no HTT, no cross-modal fusion.

This gives us a clean answer to:

> **How strong is IMU by itself?**

In [ ]:
print("device:",DEVICE)


class MambaBlock(nn.Module):
    """Pre-norm residual wrapper around the real mamba_ssm.Mamba (v28)."""
    def __init__(self):
        super().__init__()
        self.norm=nn.LayerNorm(D_MODEL)
        self.mamba=Mamba(d_model=D_MODEL,d_state=MAMBA_D_STATE,d_conv=MAMBA_D_CONV,expand=MAMBA_EXPAND)

    def forward(self,x):
        return x+self.mamba(self.norm(x))


class IMUMamba(nn.Module):
    """v28 UnimodalMamba, sensor path (S1)."""
    def __init__(self, in_dim=6, n_classes=N_CLASSES):
        super().__init__()
        self.sensor_encoder=nn.Sequential(
            nn.Conv1d(in_dim,64,kernel_size=5,padding=2), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64,D_MODEL,kernel_size=3,padding=1), nn.BatchNorm1d(D_MODEL), nn.GELU())
        self.modality_embedding=nn.Embedding(1,D_MODEL)
        self.token_norm=nn.LayerNorm(D_MODEL)
        self.mamba_blocks=nn.ModuleList([MambaBlock() for _ in range(MAMBA_NUM_BLOCKS)])
        self.pool_attn=nn.Sequential(nn.Linear(D_MODEL,64),nn.Tanh(),nn.Linear(64,1))
        self.classifier=nn.Sequential(nn.Dropout(DROPOUT),nn.Linear(D_MODEL,D_MODEL//2),nn.GELU(),
                                      nn.Linear(D_MODEL//2,n_classes))

    def forward(self,x):
        z=self.sensor_encoder(x.transpose(1,2)).transpose(1,2)
        h=self.token_norm(z+self.modality_embedding.weight.view(1,1,-1))
        for b in self.mamba_blocks:
            h=b(h)
        a=F.softmax(self.pool_attn(h),dim=1)
        return self.classifier((h*a).sum(dim=1))


def verify_model(model):
    n_mamba=sum(p.numel() for p in model.mamba_blocks.parameters())
    ok=(all(type(b.mamba) is MAMBA_CLASS for b in model.mamba_blocks) and MAMBA_CLASS.__module__.startswith("mamba_ssm")
        and len(model.mamba_blocks)==MAMBA_NUM_BLOCKS and n_mamba==EXPECTED_MAMBA_PARAMS)
    return ok, sum(p.numel() for p in model.parameters()), n_mamba


seed_all(SEEDS[0])
model=IMUMamba().to(DEVICE)
_ok,N_PARAMS,_nm=verify_model(model)
assert _ok, "model is not the expected real-mamba_ssm architecture"
print(f"parameters: {N_PARAMS:,} (Mamba blocks {_nm:,} == {EXPECTED_MAMBA_PARAMS:,}; block class {MAMBA_CLASS.__module__}.{MAMBA_CLASS.__name__})")
del model
torch.cuda.empty_cache()

## 13. Day-out evaluation

Each recording day is held out completely.

This is the same generalization question used for the audio model:

> Can IMU classify a fault on a day whose missions were never seen during training?

The folds are **identical to the audio notebook's**:
- Fold k tests on day k, in `DAYS` order.
- Inner validation holds out ~15 % of each class's training missions (at least 1). The rule is v27's
  `_split_inner_val` if `version 27/minor-project-v27.ipynb` is attached; otherwise the same per-class hold-out with
  fixed seed 2026. The audio notebook applies the same rule.
- z-scoring is per channel, from the fold's **training windows only**. Class weights also come from the training
  windows.
- Every run starts with `seed_all(seed)`. DataLoader shuffling uses a seeded generator and `worker_init_fn`.

In [ ]:
# Utility functions (seed_all is defined in the constants cell and called at the START of every run by run_one)
class WDS(Dataset):
    def __init__(self,X,y):
        self.X=torch.tensor(X,dtype=torch.float32)
        self.y=torch.tensor(y,dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self,i): return self.X[i],self.y[i]


def make_loader(X,y,shuffle,seed):
    return DataLoader(WDS(X,y),batch_size=BATCH_SIZE,shuffle=shuffle,num_workers=NUM_WORKERS,pin_memory=True,
                      worker_init_fn=worker_init_fn,generator=make_generator(seed) if shuffle else None)


def predict_model(model,X):
    dl=make_loader(X,np.zeros(len(X),dtype=np.int64),False,0)
    ps=[]
    model.eval()
    with torch.no_grad():
        for xb,_ in dl:
            ps.append(torch.softmax(model(xb.to(DEVICE)),1).cpu().numpy())
    return np.concatenate(ps)


def train_one(Xtr,ytr,Xv,yv,seed,epochs=EPOCHS,verbose=True):
    """v28 training: AdamW, cosine to LR_MIN over EPOCHS, grad clip, class-weighted CE, best validation macro-F1.
    The caller (run_one) has already called seed_all(seed)."""
    model=IMUMamba().to(DEVICE)
    init_sha1=hashlib.sha1(b"".join(v.detach().cpu().contiguous().numpy().tobytes()
                                    for v in model.state_dict().values())).hexdigest()[:12]
    assert verify_model(model)[0]
    opt=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS,eta_min=LR_MIN)
    counts=np.bincount(ytr,minlength=N_CLASSES)
    weights=counts.sum()/(N_CLASSES*np.maximum(counts,1))            # v28: inverse class frequency, training windows
    lossfn=nn.CrossEntropyLoss(weight=torch.tensor(weights,dtype=torch.float32,device=DEVICE))
    tr=make_loader(Xtr,ytr,True,seed)
    va=make_loader(Xv,yv,False,seed)
    best_state,bestf,best_epoch,history=None,-1.0,None,[]
    for ep in range(1,epochs+1):
        model.train()
        running,nb=0.0,0
        for xb,yb in tr:
            xb,yb=xb.to(DEVICE,non_blocking=True),yb.to(DEVICE,non_blocking=True)
            opt.zero_grad()
            loss=lossfn(model(xb),yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=GRAD_CLIP_NORM)
            opt.step()
            running+=loss.item(); nb+=1
        sched.step()
        p=predict_model(model,Xv)
        f=f1_score(yv,p.argmax(1),average="macro",labels=list(range(N_CLASSES)),zero_division=0)
        history.append((running/nb,f))
        if f>bestf:
            bestf,best_epoch=f,ep
            best_state=copy.deepcopy(model.state_dict())      # a real copy (a bare state_dict() would keep changing)
        if verbose:
            print(f"    epoch {ep:02d}/{epochs} | train CE {running/nb:.4f} | val macro-F1 {f:.4f}" + ("  * best" if ep==best_epoch else ""))
    model.load_state_dict(best_state)
    return model,{"history":history,"best_epoch":best_epoch,"best_val_macro_f1":bestf,"init_sha1":init_sha1,
                  "class_weights":np.round(weights,4).tolist()}


def run_one(k,seed,epochs=EPOCHS,verbose=True):
    """One run = seed_all(seed) FIRST, then fold-k train-only z-score, training and test prediction."""
    seed_all(seed)
    f=FOLDS[k]
    tr_mask=np.isin(groups,f["train"]); va_mask=np.isin(groups,f["val"]); te_mask=np.isin(groups,f["test"])
    assert set(groups[tr_mask])==set(f["train"])                       # z-score stats come from training missions only
    mu=W[tr_mask].mean(axis=(0,1),keepdims=True)
    sd=W[tr_mask].std(axis=(0,1),keepdims=True)+SENSOR_STD_EPS
    zhash=hashlib.sha1(mu.tobytes()+sd.tobytes()).hexdigest()[:12]
    model,info=train_one((W[tr_mask]-mu)/sd,y[tr_mask],(W[va_mask]-mu)/sd,y[va_mask],seed,epochs,verbose)
    p=predict_model(model,(W[te_mask]-mu)/sd)
    del model; gc.collect(); torch.cuda.empty_cache()
    return p,{**info,"zstats_sha1":zhash},te_mask,(tr_mask,va_mask)

In [ ]:
# ==============================================================================
# DAY-OUT FOLDS — identical to the audio notebook (fold k = DAYS[k]; v27 inner-validation rule, else seed 2026)
# ==============================================================================
def extract_function(nb_path,fname):
    try:
        nb=json.loads(Path(nb_path).read_text(encoding="utf-8"))
    except Exception as exc:
        print(f"  could not read {nb_path}: {exc!r}")
        return None
    for cell in nb.get("cells",[]):
        if cell.get("cell_type")!="code":
            continue
        src=cell["source"] if isinstance(cell["source"],str) else "".join(cell["source"])
        src="\n".join(l for l in src.split("\n") if not l.lstrip().startswith(("!","%")))
        try:
            tree=ast.parse(src)
        except SyntaxError:
            continue
        for node in tree.body:
            if isinstance(node,ast.FunctionDef) and node.name==fname:
                return ast.get_source_segment(src,node)
    return None


LABEL={Path(m):int(c) for m,c in zip(missions["mission"],missions["class"])}   # v27's rule keys missions by Path (.name)
_p27=find_file("minor-project-v27.ipynb",["version 27","Version 27"])
_src27=extract_function(_p27,"_split_inner_val") if _p27 else None
if _src27 is not None and hashlib.sha1(_src27.encode()).hexdigest()[:16]==V27_INNER_VAL_SHA1:
    _ns27={"np":np,"N_CLASSES":N_CLASSES,"LABEL":LABEL,"INNER_VAL_SEED":INNER_VAL_SEED,"INNER_VAL_FRACTION":INNER_VAL_FRACTION}
    exec(_src27,_ns27)
    inner_val=lambda pool,k: _ns27["_split_inner_val"](pool,0,k)
    INNER_VAL_RULE=f"v27 _split_inner_val (from {_p27}; seed [{INNER_VAL_SEED}, 0, fold]) — same as the audio notebook"
else:
    loud(("v27 notebook not found" if _p27 is None else f"v27 _split_inner_val unreadable/fingerprint mismatch at {_p27}")
         + f"\nFALLBACK inner validation: per-class ~15% (min 1) mission hold-out, fixed seed {INNER_VAL_FALLBACK_SEED} "
           "(the audio notebook's fallback, identical code).")

    def inner_val(pool,k):
        rng=np.random.default_rng([INNER_VAL_FALLBACK_SEED,k])
        train,val=[],[]
        for c in range(N_CLASSES):
            ms=sorted((m for m in pool if LABEL[m]==c),key=lambda m:m.name)
            perm=rng.permutation(len(ms))
            n_val=max(1,int(round(INNER_VAL_FRACTION*len(ms))))
            val+=[ms[i] for i in perm[:n_val]]
            train+=[ms[i] for i in perm[n_val:]]
        return train,val

    INNER_VAL_RULE=f"FALLBACK per-class hold-out, seed {INNER_VAL_FALLBACK_SEED} (same as the audio notebook's fallback)"
print("Inner-validation rule:",INNER_VAL_RULE)

FOLDS={}
_all=set(missions["mission"])
for k,day in enumerate(DAYS):
    test=sorted(missions.loc[missions["day"]==day,"mission"])
    tr,va=inner_val([Path(m) for m in sorted(_all-set(test))],k)
    FOLDS[k]={"day":day,"train":sorted(p.name for p in tr),"val":sorted(p.name for p in va),"test":test}
    f=FOLDS[k]
    assert not (set(f["train"])&set(f["val"])) and not (set(f["train"]+f["val"])&set(test))
    assert set(f["train"]+f["val"]+test)==_all
    assert day not in {imu_data[m]["day"] for m in f["train"]+f["val"]}, f"fold {k}: test day in training"
assert sorted(m for f in FOLDS.values() for m in f["test"])==sorted(_all)

print(f"{'fold':<5}{'test day':<12}{'missions tr/val/test':<22}{'test missions per class':<26}{'windows tr/val/test'}")
for k,f in FOLDS.items():
    per_class=np.bincount([imu_data[m]["class"] for m in f["test"]],minlength=N_CLASSES).tolist()
    n_w="/".join(str(int(np.isin(groups,f[s]).sum())) for s in ("train","val","test"))
    print(f"{k:<5}{f['day']:<12}{'/'.join(str(len(f[s])) for s in ('train','val','test')):<22}{str(per_class):<26}{n_w}"
          + ("   <- SMALL fold (10 missions)" if f["day"]==SMALL_FOLD_DAY else ""))

## Validation gate (part B — windows and determinism)
This part runs immediately before the day-out runs:
- Window ids are unique, and every window belongs to exactly one fold's test set. Every day-out test set is disjoint
  from its training and validation sets.
- **Determinism:** seed 42 is trained for 2 epochs, twice, on the same fold. The loss trajectories and the initial
  weights must be bit-identical. A FAIL stops the notebook (`DETERMINISM_STOP = True`).

In [ ]:
# ==============================================================================
# VALIDATION GATE — part B (windows + determinism)
# ==============================================================================
_n_test=sum(int(np.isin(groups,f["test"]).sum()) for f in FOLDS.values())
gate("window ids unique; every window in exactly one day-out test fold", len(set(window_ids))==len(window_ids)==_n_test,
     f"{len(window_ids)} windows, {_n_test} test-window slots over 5 folds")
print(f"Determinism check: fold {DETERMINISM_FOLD} ({FOLDS[DETERMINISM_FOLD]['day']}), seed 42, {DETERMINISM_EPOCHS} epochs, twice ...")
_,_i1,_,_=run_one(DETERMINISM_FOLD,42,epochs=DETERMINISM_EPOCHS,verbose=False)
_,_i2,_,_=run_one(DETERMINISM_FOLD,42,epochs=DETERMINISM_EPOCHS,verbose=False)
for e,(a,b) in enumerate(zip(_i1["history"],_i2["history"]),1):
    print(f"  epoch {e}: run 1 train CE {a[0]:.10f} val-F1 {a[1]:.6f} | run 2 train CE {b[0]:.10f} val-F1 {b[1]:.6f}")
_det=gate("determinism: identical initial weights and loss trajectories (seed 42, 2 epochs, twice)",
          _i1["history"]==_i2["history"] and _i1["init_sha1"]==_i2["init_sha1"],
          f"init {_i1['init_sha1']} vs {_i2['init_sha1']}")
if not _det and DETERMINISM_STOP:
    loud("DETERMINISM CHECK FAILED: two identical seed-42 runs gave different losses.\n"
         "mamba-ssm's CUDA backward may be non-deterministic on this GPU/version. Set DETERMINISM_STOP = False in the\n"
         "constants cell to continue anyway (results would then be reproducible only up to that noise).")
    raise RuntimeError("Validation gate (part B) failed — determinism.")
print("Validation gate: " + ("all PASS" if all(ok for _,ok,_ in GATE) else "FAIL present (see above)"))

In [ ]:
# Run day-out folds: 5 folds x 3 seeds. Every run starts with seed_all(seed) (inside run_one).
# Per-window probabilities -> imu_only_probs/day_out_{fold}_{seed}.npz (the audio notebook's schema);
# one row per run -> imu_only_results.csv.
if IMU_RESULTS_CSV.exists():
    IMU_RESULTS_CSV.unlink()                     # a fresh Run All writes a fresh results log
CSV_FIELDS=["protocol","fold","fold_day","seed","n_train_missions","n_val_missions","n_test_missions","n_train_windows",
            "n_val_windows","n_test_windows","accuracy","macro_f1","balanced_accuracy","macro_auroc","mission_accuracy",
            "mission_macro_f1","mission_balanced_accuracy","best_epoch","best_val_macro_f1","init_sha1","zstats_sha1",
            "class_weights","n_params","train_seconds","probs_file","finished_at"]

all_pred=[]
fold_summary=[]
_t0=time.time(); _n=0; _N=len(FOLDS)*len(SEEDS)

for k,f in FOLDS.items():
    fold_day=f["day"]
    for seed in SEEDS:
        t_run=time.time()
        print(f"[run {_n+1:02d}/{_N}] day-out fold {k} ({fold_day}) seed {seed}")
        p,info,te_mask,(tr_mask,va_mask)=run_one(k,seed)
        pred=p.argmax(1)
        yt=y[te_mask]
        mdf=pd.DataFrame(p); mdf["mission"]=groups[te_mask]; mdf["y"]=yt
        mg=mdf.groupby("mission"); my=mg["y"].first().values; mP=mg[list(range(N_CLASSES))].mean().values
        try:
            auroc=roc_auc_score(yt,p,multi_class="ovr",average="macro",labels=list(range(N_CLASSES)))
        except ValueError:
            auroc=np.nan
        row={"protocol":"day_out","fold":k,"fold_day":fold_day,"seed":seed,
             "n_train_missions":len(f["train"]),"n_val_missions":len(f["val"]),"n_test_missions":len(f["test"]),
             "n_train_windows":int(tr_mask.sum()),"n_val_windows":int(va_mask.sum()),"n_test_windows":int(te_mask.sum()),
             "accuracy":accuracy_score(yt,pred),"macro_f1":f1_score(yt,pred,average="macro",labels=list(range(N_CLASSES)),zero_division=0),
             "balanced_accuracy":balanced_accuracy_score(yt,pred),"macro_auroc":auroc,
             "mission_accuracy":accuracy_score(my,mP.argmax(1)),
             "mission_macro_f1":f1_score(my,mP.argmax(1),average="macro",labels=list(range(N_CLASSES)),zero_division=0),
             "mission_balanced_accuracy":balanced_accuracy_score(my,mP.argmax(1)),
             "best_epoch":info["best_epoch"],"best_val_macro_f1":round(info["best_val_macro_f1"],6),
             "init_sha1":info["init_sha1"],"zstats_sha1":info["zstats_sha1"],"class_weights":json.dumps(info["class_weights"]),
             "n_params":N_PARAMS,"train_seconds":round(time.time()-t_run,1),
             "probs_file":str(IMU_PROBS_DIR/f"day_out_{k}_{seed}.npz"),"finished_at":datetime.now().isoformat(timespec="seconds")}
        np.savez(IMU_PROBS_DIR/f"day_out_{k}_{seed}.npz",window_id=window_ids[te_mask].astype(str),
                 mission=groups[te_mask].astype(str),cls=yt,day=days[te_mask].astype(str),phase=phases[te_mask].astype(str),
                 start=starts_s[te_mask],probs=p.astype(np.float32))
        with open(IMU_RESULTS_CSV,"a",newline="") as fh:
            wr=csv.DictWriter(fh,fieldnames=CSV_FIELDS)
            if fh.tell()==0:
                wr.writeheader()
            wr.writerow(row)
        fold_summary.append({"fold":k,"test_day":fold_day,"seed":seed,"macro_f1":row["macro_f1"],
                             "balanced_acc":row["balanced_accuracy"],"accuracy":row["accuracy"]})
        for idx,prob in zip(np.where(te_mask)[0],p):
            all_pred.append({
                "window_idx":int(idx),
                "mission":groups[idx],
                "class":int(y[idx]),
                "day":days[idx],
                "seed":seed,
                "prob":prob
            })
        _n+=1
        print(f"[run {_n:02d}/{_N}] done | window macro-F1 {row['macro_f1']:.4f} | mission macro-F1 {row['mission_macro_f1']:.4f} "
              f"| best epoch {info['best_epoch']} | {row['train_seconds']:.0f} s | ETA {(_N-_n)*(time.time()-_t0)/_n/60:.0f} min")

fold_summary=pd.DataFrame(fold_summary)
_fs=fold_summary.groupby(["fold","test_day"])[["macro_f1","balanced_acc","accuracy"]].mean().reset_index()
_fs["note"]=np.where(_fs.test_day==SMALL_FOLD_DAY,"SMALL fold (10 missions)","")
print("Per fold (mean over seeds):")
display(_fs)
display(fold_summary.groupby("seed")["macro_f1"].mean())

## 14. Pooled day-out IMU performance

Each window's probabilities are averaged over the 3 seeds. Every window is tested exactly once, on a model that never
saw its recording day.
- **95 % CIs** come from a **mission-level cluster bootstrap**: 2000 resamples of missions within class, never
  windows. This is the audio notebook's method. They are reported for every per-class precision/recall/F1 and for
  pooled macro-F1.
- **Per fold,** only macro-F1 and balanced accuracy are reported. Per-fold per-class recall is not reported: each day
  has only 2–8 missions per class. The 2021-12-22 fold (10 missions) is flagged as small.
- **Safety-relevant errors:** faulty windows called Normal, Normal windows called faulty, and adjacent-class vs
  distant errors.

In [ ]:
# Average probabilities across seeds for each test window
_pm=defaultdict(list)
for r in all_pred:
    _pm[r["window_idx"]].append(r["prob"])
pred_rows=[]
for idx in sorted(_pm):
    assert len(_pm[idx])==len(SEEDS), f"window {idx}: {len(_pm[idx])} seeds"
    prob=np.mean(_pm[idx],axis=0)
    pred_rows.append({
        "window_idx":idx,
        "window_id":window_ids[idx],
        "mission":groups[idx],
        "day":days[idx],
        "phase":phases[idx],
        "true":int(y[idx]),
        "pred":int(np.argmax(prob)),
        "prob":prob
    })
pred_df=pd.DataFrame(pred_rows)
assert len(pred_df)==len(windows), "every window must be tested exactly once per seed"


def prf_from_cm(cm):
    tp=np.diag(cm).astype(float)
    prec=np.divide(tp,cm.sum(0),out=np.zeros_like(tp),where=cm.sum(0)>0)
    rec=np.divide(tp,cm.sum(1),out=np.zeros_like(tp),where=cm.sum(1)>0)
    f1=np.divide(2*prec*rec,prec+rec,out=np.zeros_like(tp),where=(prec+rec)>0)
    return prec,rec,f1


def cluster_bootstrap(mission_ids,yt,yp,B=BOOTSTRAP_RESAMPLES,seed=BOOTSTRAP_SEED):
    """Resample MISSIONS within class (never windows); returns B bootstrap confusion matrices."""
    rng=np.random.default_rng(seed)
    ms=pd.Series(mission_ids).unique()
    pos={m:i for i,m in enumerate(ms)}
    cnt=np.zeros((len(ms),N_CLASSES),int); mcls=np.zeros(len(ms),int)
    for m,t,p in zip(mission_ids,yt,yp):
        cnt[pos[m],p]+=1; mcls[pos[m]]=t
    cms=np.zeros((B,N_CLASSES,N_CLASSES))
    for c in range(N_CLASSES):
        rows_c=np.where(mcls==c)[0]
        draw=rows_c[rng.integers(0,len(rows_c),(B,len(rows_c)))]
        cms[:,c,:]=cnt[draw].sum(1)
    return cms


def boot_report(mission_ids,yt,yp,level):
    cm=confusion_matrix(yt,yp,labels=list(range(N_CLASSES)))
    p,r,f=prf_from_cm(cm)
    bp,br,bf=map(np.array,zip(*(prf_from_cm(c) for c in cluster_bootstrap(mission_ids,yt,yp))))
    ci=lambda a: np.percentile(a,[2.5,97.5],axis=0)
    tab=pd.DataFrame({"precision":p,"P 95% CI":[f"[{a:.3f}, {b:.3f}]" for a,b in ci(bp).T],
                      "recall":r,"R 95% CI":[f"[{a:.3f}, {b:.3f}]" for a,b in ci(br).T],
                      "f1":f,"F1 95% CI":[f"[{a:.3f}, {b:.3f}]" for a,b in ci(bf).T],"support":cm.sum(1)},
                     index=[f"class {c}" for c in range(N_CLASSES)])
    mlo,mhi=ci(bf.mean(1))
    print(f"{level}: macro-F1 {f.mean():.4f} [95% CI {mlo:.4f}, {mhi:.4f}] | balanced accuracy {r.mean():.4f} | "
          f"accuracy {accuracy_score(yt,yp):.4f}")
    display(tab.round(3))
    return {"macro_f1":f.mean(),"ci":(mlo,mhi),"table":tab,"cm":cm}


WIN_RES=boot_report(pred_df.mission.values,pred_df.true.values,pred_df.pred.values,
                    "Pooled day-out, WINDOW level (seed-averaged)")
print("Confusion matrix (rows = true, cols = predicted):")
print(WIN_RES["cm"])
_P=np.stack(pred_df.prob.values)
print("Window macro-AUROC:",roc_auc_score(pred_df.true,_P,multi_class="ovr",average="macro"))

_seed_f1={}
for s in SEEDS:
    ps=np.stack([r["prob"] for r in sorted((r for r in all_pred if r["seed"]==s),key=lambda r:r["window_idx"])])
    _seed_f1[s]=f1_score(pred_df.true,ps.argmax(1),average="macro",labels=list(range(N_CLASSES)),zero_division=0)
print("Per-seed pooled window macro-F1:",{s:round(v,4) for s,v in _seed_f1.items()},
      f"| seed-to-seed SD {np.std(list(_seed_f1.values()),ddof=1):.4f}")

print("\nPer fold (= unseen day), seed-averaged probabilities — macro-F1 and balanced accuracy only:")
_pf=[]
for k,f in FOLDS.items():
    g=pred_df[pred_df.day==f["day"]]
    _pf.append({"fold":k,"test day":f["day"],"missions":g.mission.nunique(),"windows":len(g),
                "macro-F1":f1_score(g.true,g.pred,average="macro",labels=list(range(N_CLASSES)),zero_division=0),
                "balanced acc":balanced_accuracy_score(g.true,g.pred),
                "note":"SMALL fold (10 missions)" if f["day"]==SMALL_FOLD_DAY else ""})
display(pd.DataFrame(_pf).set_index("fold").round(4))

# ---- safety-relevant breakdown (window level, pooled) ----
_t,_p=pred_df.true.values,pred_df.pred.values
_err=_p!=_t
SAFETY={"faulty windows predicted Normal (missed fault)":float((_p[_t>0]==0).mean()),
        "Normal windows predicted faulty (false alarm)":float((_p[_t==0]>0).mean()),
        "errors that are adjacent-class (|pred-true|==1)":float((np.abs(_p-_t)[_err]==1).mean()) if _err.any() else np.nan,
        "errors that are distant (|pred-true|>=2)":float((np.abs(_p-_t)[_err]>=2).mean()) if _err.any() else np.nan}
print("\nSafety-relevant errors (day-out pooled, window level):")
for k_,v in SAFETY.items():
    print(f"  {k_:<52} {v:.3f}")

## 15. Mission-level IMU performance

Multiple windows from the same mission are averaged before classification.
This is directly comparable to the audio mission-level result. The mission prediction is the argmax of the mean
window probability, as in the audio notebook.

The mission-level day-out predictions are saved to `imu_only_mission_predictions.csv`, with columns mission, true,
pred and p0–p4.

In [ ]:
mission_probs={}
for m,g in pred_df.groupby("mission"):
    probs=np.stack(g["prob"].values)
    mission_probs[m]={
        "true":int(g.true.iloc[0]),
        "pred":int(np.argmax(probs.mean(axis=0))),
        "prob":probs.mean(axis=0)
    }

mp=pd.DataFrame([
    {"mission":m,**{k:v for k,v in d.items() if k!="prob"},**{f"p{c}":float(d["prob"][c]) for c in range(N_CLASSES)}}
    for m,d in mission_probs.items()
])
assert len(mp)==EXPECTED_N_MISSIONS
MISSION_RES=boot_report(mp.mission.values,mp.true.values,mp.pred.values,"Pooled day-out, MISSION level")
print("Mission accuracy:",accuracy_score(mp.true,mp.pred))
mp[["mission","true","pred"]+[f"p{c}" for c in range(N_CLASSES)]].to_csv(IMU_MISSION_CSV,index=False)
print("saved:",IMU_MISSION_CSV)

## 16. Does IMU specifically solve the audio's hard cases?

This is the critical section.

Earlier audio analyses found the main audio errors at:

- 2 ↔ 3
- 3 ↔ 4

We ask:

1. How often does IMU get an audio mistake correct?
2. How often do both modalities make the same mistake?
3. What is the oracle union accuracy — correct if either modality is correct?

This is **not yet a fusion model**. It is a complementarity diagnostic.

**Audio input.** This section reads the audio notebook's **saved per-window day-out probabilities**,
`day_out_{fold}_{seed}.npz`. It looks in three places, in order:
- `/kaggle/working/audio_only_probs/`, if both notebooks ran in the same session;
- any attached `/kaggle/input/**/audio_only_probs/` folder;
- the project repo cloned in the setup cell, under `Audio Analysis /Analysis/PT and npz/`.

It then does the following:
- averages each window over the seeds found;
- averages the windows of each mission;
- takes the argmax as the audio mission prediction.

If no such file is found, it prints the paths it checked and skips this section; nothing fails.

In [ ]:
# Load the audio notebook's saved day-out per-window probabilities and build audio mission-level predictions.
AUDIO_PROB_DIRS=[WORK/"audio_only_probs",Path.cwd()/"audio_only_outputs"/"audio_only_probs"]
AUDIO_PROB_DIRS+=[Path(p) for p in sorted(glob.glob("/kaggle/input/**/audio_only_probs",recursive=True))]
AUDIO_PROB_DIRS+=[REPO_AUDIO_PROBS]      # the audio run's saved probabilities, pushed to the project repo
audio_files=[]
for _d in AUDIO_PROB_DIRS:
    audio_files=sorted(_d.glob("day_out_*.npz")) if _d.is_dir() else []
    if audio_files:
        AUDIO_DIR=_d
        break

audio_mp=None
AUDIO_REF=None
if not audio_files:
    print("No audio day-out probabilities found. Checked:")
    for _d in AUDIO_PROB_DIRS:
        print("   ",_d/"day_out_*.npz")
    print("Section 16 is SKIPPED: no audio reference available. Run TII_Audio_Only_Analysis_Mamba.ipynb first "
          "(same session), or attach its audio_only_probs/ folder as an input dataset.")
else:
    parts=[]
    for fp in audio_files:
        z=np.load(fp)
        seed=int(fp.stem.split("_")[-1])
        parts.append(pd.DataFrame({"window_id":z["window_id"],"mission":z["mission"],"cls":z["cls"],"seed":seed,
                                   **{f"p{c}":z["probs"][:,c] for c in range(N_CLASSES)}}))
    adf=pd.concat(parts,ignore_index=True)
    _pc=[f"p{c}" for c in range(N_CLASSES)]
    print(f"Loaded {len(audio_files)} audio files from {AUDIO_DIR}: seeds {sorted(adf.seed.unique())}, "
          f"{adf.window_id.nunique()} windows, {adf.mission.nunique()} missions")
    _per_w=adf.groupby("window_id").agg(mission=("mission","first"),cls=("cls","first"),n_seeds=("seed","nunique"),
                                        **{c:(c,"mean") for c in _pc})
    if _per_w.n_seeds.nunique()>1:
        loud(f"Audio windows have unequal seed coverage {sorted(_per_w.n_seeds.unique())}: some runs are missing.")
    _am=_per_w.groupby("mission").agg(true=("cls","first"),**{c:(c,"mean") for c in _pc})
    _am["pred"]=_am[_pc].values.argmax(1)
    audio_mp=_am.reset_index()
    _missing=set(mp.mission)-set(audio_mp.mission)
    if _missing:
        loud(f"{len(_missing)} IMU missions have no audio prediction; the comparison uses the {len(set(mp.mission)&set(audio_mp.mission))} shared missions.")
    _w_pred=_per_w[_pc].values.argmax(1)
    AUDIO_REF={"source":str(AUDIO_DIR),"seeds":sorted(adf.seed.unique().tolist()),"n_missions":len(audio_mp),
               "window_macro_f1":f1_score(_per_w.cls,_w_pred,average="macro",labels=list(range(N_CLASSES)),zero_division=0),
               "mission_macro_f1":f1_score(audio_mp.true,audio_mp.pred,average="macro",labels=list(range(N_CLASSES)),zero_division=0),
               "mission_accuracy":accuracy_score(audio_mp.true,audio_mp.pred)}
    print(f"Audio (loaded at run time): window macro-F1 {AUDIO_REF['window_macro_f1']:.4f} | mission macro-F1 "
          f"{AUDIO_REF['mission_macro_f1']:.4f} | mission accuracy {AUDIO_REF['mission_accuracy']:.4f}")
    display(audio_mp.head())

In [ ]:
# Complementarity at mission level (IMU = this notebook, audio = loaded above)
if audio_mp is not None:
    merged=mp[["mission","true","pred"]].merge(audio_mp[["mission","true","pred"]],on="mission",suffixes=("_imu","_audio"))
    assert (merged.true_imu==merged.true_audio).all(), "audio and IMU disagree on a mission's true class"
    merged["true"]=merged.true_imu
    merged["imu_correct"]=merged.pred_imu==merged.true
    merged["audio_correct"]=merged.pred_audio==merged.true
    merged["both_correct"]=merged.audio_correct & merged.imu_correct
    merged["audio_only_correct"]=merged.audio_correct & ~merged.imu_correct
    merged["imu_only_correct"]=~merged.audio_correct & merged.imu_correct
    merged["both_wrong"]=~merged.audio_correct & ~merged.imu_correct
    print(f"{len(merged)} missions compared")
    print(merged[["both_correct","audio_only_correct","imu_only_correct","both_wrong"]].mean())
    print("Oracle either correct:", (merged.audio_correct | merged.imu_correct).mean())
    _aw=merged[~merged.audio_correct]
    if len(_aw):
        print(f"IMU correctly resolves {_aw.imu_correct.mean():.3f} of audio's {len(_aw)} mission errors; "
              f"both make the SAME wrong prediction on {(_aw.pred_imu==_aw.pred_audio).sum()} of them")

    hard=merged[merged["true"].isin([2,3,4])]
    print("\nHard severity subset (true class 2/3/4):")
    print(hard[["both_correct","audio_only_correct","imu_only_correct","both_wrong"]].mean())
else:
    print("Section 16 skipped: no audio reference available.")

## 17. What to look for

After running the notebook, focus on these outputs:

### A. IMU standalone
Compare its day-out mission macro-F1 with the audio notebook's. The table below prints the audio number **only if it
was loaded at run time** from the audio notebook's saved probabilities (section 16). Otherwise it prints "no audio
reference available" and leaves the audio column blank. No audio number is hardcoded here.

### B. IMU class pattern
Look at the IMU confusion matrix.

If IMU also mainly makes:
`2↔3` and `3↔4`,
then it is probably measuring a related severity signal.

If its errors differ from audio's, that is much more interesting.

### C. Complementarity
The most important numbers are:

- `IMU-only correct`
- `Audio-only correct`
- `Both wrong`
- `Oracle either correct`

Especially for true classes 2–4.

### D. Architecture clue

If IMU contains strong vibration-frequency structure, a sensible later fusion design is likely:

```text
Audio → spectral encoder ─┐
                          ├→ fusion → classifier
IMU   → temporal/vibration encoder ─┘
```

rather than forcing both modalities through the same representation.

If IMU's useful information is mostly temporal vibration patterns, then Mamba has a clearer role on the IMU side than it did on the audio side.

### Important stopping rule

Do not change AeroSync-Mamba yet.

First determine **what information IMU actually contains** and whether it complements audio.

In [ ]:
# Side by side: IMU (this notebook) vs audio (ONLY if loaded at run time in section 16)
_rows=[{"metric":"day-out pooled window macro-F1","IMU":f"{WIN_RES['macro_f1']:.4f} [{WIN_RES['ci'][0]:.4f}, {WIN_RES['ci'][1]:.4f}]"},
       {"metric":"day-out mission macro-F1","IMU":f"{MISSION_RES['macro_f1']:.4f} [{MISSION_RES['ci'][0]:.4f}, {MISSION_RES['ci'][1]:.4f}]"},
       {"metric":"day-out mission accuracy","IMU":f"{accuracy_score(mp.true,mp.pred):.4f}"}]
if AUDIO_REF is None:
    print("no audio reference available")
    for r in _rows:
        r["audio (loaded)"]=""
else:
    _rows[0]["audio (loaded)"]=f"{AUDIO_REF['window_macro_f1']:.4f}"
    _rows[1]["audio (loaded)"]=f"{AUDIO_REF['mission_macro_f1']:.4f}"
    _rows[2]["audio (loaded)"]=f"{AUDIO_REF['mission_accuracy']:.4f}"
    print(f"audio reference loaded from {AUDIO_REF['source']} (seeds {AUDIO_REF['seeds']}, {AUDIO_REF['n_missions']} missions); "
          "audio CIs are in the audio notebook's section 7b")
display(pd.DataFrame(_rows).set_index("metric"))
print("\nValidation gate summary:")
for n,ok,d in GATE:
    print(f"  {'PASS' if ok else 'FAIL'} — {n}")

## Expected final conclusion format

After execution, summarize the notebook in four lines:

1. **Dominant IMU pattern:** ...
2. **Standalone day-out performance:** ... window / ... mission Macro-F1
3. **Main IMU confusions:** ...
4. **Audio rescue:** IMU correctly resolves ...% of audio's hard errors; oracle either-modality performance = ...

That will tell us whether the multimodal architecture has a real reason to exist before we touch LACA again.
